# v6 overlap40：CMCR 模块与 Gated 多种子并行实验

账号1运行 Gated+CMCR seed42，账号2运行当前 Gated seed1337 稳定性复验。两者都使用 physical batch=4、boundary weight=0，只使用 Train/Val 并按 `val_mIoU_fg` 选模，不自动查看 Test。

## 1. 选择实验


In [ ]:
# 账号1：Gated + CMCR，seed42。
CONFIG_FILE = 'v6_overlap40_gated_cmcr_batch4_seed42.json'
# 账号2：注释上一行，启用下一行的 Gated seed1337 复验。
# CONFIG_FILE = 'v6_overlap40_gated_no_boundary_batch4_seed1337.json'

REPO_URL = 'https://github.com/song110585-cpu/lunar-linear.git'
REPO_BRANCH = 'test-new-module'


## 2. 环境与代码


In [ ]:
import hashlib, importlib.metadata, importlib.util, json, shutil, subprocess, sys
from pathlib import Path

REPO_DIR = Path('/kaggle/working/lunar-linear')
PROJECT_DIR = REPO_DIR / 'LTL-Net'
OUTPUT_ROOT = Path('/kaggle/working')

required = [('rasterio', 'rasterio'), ('tqdm', 'tqdm')]
missing = [package for module, package in required if importlib.util.find_spec(module) is None]
try:
    smp_version = importlib.metadata.version('segmentation-models-pytorch')
except importlib.metadata.PackageNotFoundError:
    smp_version = None
if smp_version != '0.5.0':
    missing.append('segmentation-models-pytorch==0.5.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])

import torch
assert torch.cuda.is_available(), '请在 Notebook settings 中开启 GPU'
if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)])
else:
    subprocess.check_call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
config_path = PROJECT_DIR / 'configs' / CONFIG_FILE
config = json.loads(config_path.read_text(encoding='utf-8'))
assert config['module'] in {'gated_boundary', 'gated_cmcr'}
assert config['batch_size'] == 4
assert config['accum_steps'] == 1
assert config['boundary_weight'] == 0.0
assert config['selection_metric'] == 'val_mIoU_fg'
assert config['automatic_test_evaluation'] is False
print('GPU:', torch.cuda.get_device_name(0))
print('Commit:', commit)
print(json.dumps(config, ensure_ascii=False, indent=2))


## 3. 核验 overlap40 数据


In [ ]:
import numpy as np

existing = [Path(path) for path in config['data_candidates'] if Path(path).is_dir()]
assert existing, '没有找到 overlap40 数据集，请先 Add Data: yuanssy/datav6-overlap40'
DATA_ROOT = existing[0]
for split, expected in config['expected_tiles'].items():
    images = sorted([*(DATA_ROOT / split / 'image').glob('*.tif'), *(DATA_ROOT / split / 'image').glob('*.tiff')])
    masks = sorted([*(DATA_ROOT / split / 'mask').glob('*.tif'), *(DATA_ROOT / split / 'mask').glob('*.tiff')])
    assert len(images) == len(masks) == expected, (split, len(images), len(masks))
    assert {p.stem for p in images} == {p.stem for p in masks}, f'{split} image-mask 不匹配'
    print(split, len(images))
for name, expected_hash in config['expected_metadata_sha256'].items():
    actual_hash = hashlib.sha256((DATA_ROOT / name).read_bytes()).hexdigest()
    assert actual_hash == expected_hash, (name, actual_hash, expected_hash)
stats = json.loads((DATA_ROOT / 'normalization_stats.json').read_text(encoding='utf-8'))
assert np.allclose(stats['mean'], config['expected_mean'], rtol=0, atol=1e-12)
assert np.allclose(stats['std'], config['expected_std'], rtol=0, atol=1e-12)
print('Data:', DATA_ROOT)
print('数据版本核验通过')


## 4. 模型前向与反向冒烟测试


In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))
from models.module_models import build_module_model
from train_module_experiment import ExperimentLoss, model_outputs

smoke_model = build_module_model(config['module'], encoder_weights=None).cuda().train()
criterion = ExperimentLoss(config['module'], config['boundary_weight']).cuda()
smoke_optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=config['learning_rate'])
smoke_scaler = torch.amp.GradScaler('cuda')
smoke_batch = config['batch_size']
smoke_size = 512
smoke_x = torch.randn(smoke_batch, 5, smoke_size, smoke_size, device='cuda')
smoke_label = torch.randint(0, 5, (smoke_batch, smoke_size, smoke_size), device='cuda')
with torch.amp.autocast('cuda'):
    with_aux = config['module'] in {'gated_boundary', 'gated_cmcr'}
    logits, boundary_logits = model_outputs(smoke_model, smoke_x, with_aux)
    smoke_loss, _ = criterion(logits, smoke_label, boundary_logits)
smoke_scaler.scale(smoke_loss).backward()
smoke_scaler.step(smoke_optimizer)
smoke_scaler.update()
assert logits.shape == (smoke_batch, 5, smoke_size, smoke_size)
print('Parameters:', f'{sum(p.numel() for p in smoke_model.parameters()):,}')
print('Smoke output:', tuple(logits.shape), 'loss:', float(smoke_loss))
del smoke_model, criterion, smoke_optimizer, smoke_scaler, smoke_x, smoke_label, logits, boundary_logits, smoke_loss
torch.cuda.empty_cache()


## 5. 正式训练


In [ ]:
result_dir = OUTPUT_ROOT / f"result_{config['run_name']}"
assert not result_dir.exists(), f'结果目录已存在；如需重跑请修改 config 中的 run_name: {result_dir}'
command = [
    sys.executable, str(PROJECT_DIR / 'scripts' / 'train_module_experiment.py'),
    '--module', config['module'],
    '--data-dir', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--run-name', config['run_name'],
    '--seed', str(config['seed']),
    '--epochs', str(config['epochs']),
    '--batch-size', str(config['batch_size']),
    '--accum-steps', str(config['accum_steps']),
    '--num-workers', str(config['num_workers']),
    '--learning-rate', str(config['learning_rate']),
    '--boundary-weight', str(config['boundary_weight']),
    '--encoder-weights', config.get('encoder_weights', 'imagenet'),
]
print(' '.join(command))
print('Output:', result_dir)
subprocess.check_call(command, cwd=PROJECT_DIR)


## 6. 查看并打包验证结果


In [ ]:
metrics_path = result_dir / 'metrics.json'
checkpoint_path = result_dir / 'best_model.pth'
assert metrics_path.is_file() and checkpoint_path.is_file()
result = json.loads(metrics_path.read_text(encoding='utf-8'))
assert result['selection_metric'] == 'val_mIoU_fg'
assert result['test_evaluated'] is False
print(json.dumps(result, ensure_ascii=False, indent=2))
archive_path = shutil.make_archive(str(OUTPUT_ROOT / result_dir.name), 'zip', root_dir=result_dir)
print('请下载:', archive_path)
